# Bootstrap Methodology: Justifying the Statistical-Testing Choices


This notebook evaluates the three trained backbones (DINOv2 ViT-B/14, ResNet50,
timm ViT-B/16) on the held-out test set, computes per-model F1, and uses the
results to justify three choices made in the main statistical-validity analysis:

1. **Minimum camera size (`MIN_CAM_IMAGES = 15`)** — per-camera F1 variance drops
   and stabilizes once a camera has at least this many images.
2. **Bootstrap resample count (`N_BOOTSTRAP = 2000`)** — the 95% CI width converges
   well before 2000 resamples, so this is more than sufficient.
3. **Cluster (camera-level) vs. image-level bootstrap** — images from the same
   camera are spatially/temporally correlated, so resampling cameras rather than
   images gives properly wider, more honest confidence intervals. This notebook
   quantifies exactly how much wider, on the same filtered population used for
   both bootstraps.

All three are combined into a single figure
(`evaluation_outputs/methodological_justification.{png,pdf}`), with its LaTeX
`\caption` printed at the end of the notebook.


In [ ]:
import sys
from collections import Counter
from pathlib import Path

import yaml

sys.path.insert(0, "../../src")

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import f1_score
from tqdm import tqdm

from data.iwildcam import iWildCamDataModule
from models.base_classifier import BaseClassifier

OUTPUT_DIR = Path("./evaluation_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def _best_checkpoint(base: Path, model_dir: str) -> str:
    """Resolve the winning LR's checkpoint via results.yaml instead of hardcoding it."""
    sweep_dir = base / f"sweep_{model_dir}"
    with open(sweep_dir / "results.yaml") as f:
        results = yaml.safe_load(f)
    lr_dir = sweep_dir / f"lr_{results['best_lr']:.0e}"
    ckpts = [p for p in lr_dir.glob("*.ckpt") if p.name != "last.ckpt"]
    assert len(ckpts) == 1, (
        f"expected exactly one checkpoint in {lr_dir}, found {ckpts}"
    )
    return str(ckpts[0])


BASE = Path("../../checkpoints/cycle1_sweep")

MODELS = {
    "DINOv2 ViT-B/14": _best_checkpoint(BASE, "dinov2_vitb14"),
    "ResNet50 pretrained": _best_checkpoint(BASE, "resnet50_pretrained"),
    "timm ViT-B/16": _best_checkpoint(BASE, "timm_vit_base_patch16_224"),
}

MODEL_A = "DINOv2 ViT-B/14"

MIN_CAM_IMAGES = 15

# Efron, B. & Tibshirani, R.J. (1993), An Introduction to the Bootstrap
N_BOOTSTRAP = 2000
SEED = 42
rng = np.random.default_rng(SEED)

In [ ]:
dm = iWildCamDataModule(
    root_dir=Path("/path/to/iwildcam_224/"),
    metadata_path=Path("metadata/metadata/iwildcam2022_train_annotations.json"),
    image_dir="train/train",
    batch_size=64,
    num_workers=4,
)
dm.setup("test")

full_ds = dm.test_dataset.dataset
test_idx_list = list(dm.test_dataset.indices)

sample_locations = []
for idx in test_idx_list:
    img_id = full_ds.valid_image_ids[idx]
    sample_locations.append(full_ds.image_lookup[img_id]["location"])
sample_locations = np.array(sample_locations)

train_class_dist = dm.train_class_dist
test_class_dist = dm.test_class_dist
eligible_classes = set(train_class_dist) & set(test_class_dist)

cam_counts = Counter(sample_locations)
qualifying_cams = {cam for cam, n in cam_counts.items() if n >= MIN_CAM_IMAGES}
print(f"Total val cameras: {len(cam_counts)}")
print(f"Qualifying cameras (>={MIN_CAM_IMAGES} images): {len(qualifying_cams)}")
print(
    f"Images in qualifying cameras: {sum(cam_counts[c] for c in qualifying_cams)} "
    f"({sum(cam_counts[c] for c in qualifying_cams) / len(sample_locations):.1%} of val)"
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_loader = dm.test_dataloader()

# Get predictions for each model.
all_results = {}
for model_name, ckpt_path in MODELS.items():
    print(f"\nEvaluating {model_name}...")
    model = BaseClassifier.load_from_checkpoint(ckpt_path)
    model.eval().to(device)

    preds_list, targets_list = [], []
    with torch.no_grad():
        for images, targets in tqdm(test_loader, desc=model_name):
            logits = model(images.to(device))
            preds_list.extend(logits.argmax(1).cpu().tolist())
            targets_list.extend(targets.tolist())

    preds = np.array(preds_list)
    targets = np.array(targets_list)

    mask = np.isin(targets, list(eligible_classes))
    preds_e = preds[mask]
    targets_e = targets[mask]
    locs_e = sample_locations[mask]

    all_results[model_name] = dict(
        preds=preds_e,
        targets=targets_e,
        locations=locs_e,
    )

    f1m = f1_score(targets_e, preds_e, average="macro", zero_division=0)
    print(f"  F1 macro (eligible classes): {f1m:.4f}")
    del model
    torch.cuda.empty_cache()

In [ ]:
# Run a single large bootstrap (5000), then compute CIs at progressive
# subsample sizes to show convergence

print("=" * 70)
print("BOOTSTRAP CONVERGENCE ANALYSIS")
print("=" * 70)

N_LARGE = 5000
res_a = all_results[MODEL_A]

cameras = np.unique(res_a["locations"])
cam_idx = {cam: np.where(res_a["locations"] == cam)[0] for cam in cameras}

# Generate all 5000 bootstrap samples
convergence_rng = np.random.default_rng(SEED)
boot_f1_large = []

for _ in tqdm(range(N_LARGE), desc="Convergence bootstrap"):
    sampled_cams = convergence_rng.choice(cameras, size=len(cameras), replace=True)
    idx = np.concatenate([cam_idx[c] for c in sampled_cams])
    boot_f1_large.append(
        f1_score(
            res_a["targets"][idx], res_a["preds"][idx], average="macro", zero_division=0
        )
    )

boot_f1_large = np.array(boot_f1_large)

# Evaluate CI at progressive subsample sizes
checkpoints = [100, 200, 500, 750, 1000, 1500, 2000, 3000, 4000, 5000]

print(
    f"\n{'n_boot':>8} {'CI low':>10} {'CI high':>10} {'CI width':>10} "
    f"{'Mean':>10} {'Std':>10}"
)
print("-" * 65)

convergence_data = []
for n in checkpoints:
    sub = boot_f1_large[:n]
    ci = np.percentile(sub, [2.5, 97.5])
    convergence_data.append(
        {
            "n": n,
            "ci_low": ci[0],
            "ci_high": ci[1],
            "width": ci[1] - ci[0],
            "mean": sub.mean(),
            "std": sub.std(),
        }
    )
    print(
        f"{n:>8} {ci[0]:>10.4f} {ci[1]:>10.4f} {ci[1] - ci[0]:>10.4f} "
        f"{sub.mean():>10.4f} {sub.std():>10.4f}"
    )

#    measure CI *of the CI* (how stable is the interval itself?)
print("\n── CI stability (10 independent runs per size) ──")
print(
    f"{'n_boot':>8} {'Mean width':>12} {'Std width':>12} {'Max |Δ low|':>12} {'Max |Δ high|':>12}"
)
print("-" * 60)

stability_data = []
for n in [500, 1000, 2000, 5000]:
    widths = []
    lows = []
    highs = []
    for rep in range(10):
        rep_rng = np.random.default_rng(SEED + rep + 1)
        rep_samples = []
        for _ in range(n):
            sampled_cams = rep_rng.choice(cameras, size=len(cameras), replace=True)
            idx = np.concatenate([cam_idx[c] for c in sampled_cams])
            rep_samples.append(
                f1_score(
                    res_a["targets"][idx],
                    res_a["preds"][idx],
                    average="macro",
                    zero_division=0,
                )
            )
        rep_samples = np.array(rep_samples)
        ci = np.percentile(rep_samples, [2.5, 97.5])
        widths.append(ci[1] - ci[0])
        lows.append(ci[0])
        highs.append(ci[1])

    widths = np.array(widths)
    lows = np.array(lows)
    highs = np.array(highs)
    stability_data.append(
        {
            "n": n,
            "mean_width": widths.mean(),
            "std_width": widths.std(),
            "max_delta_low": np.ptp(lows),
            "max_delta_high": np.ptp(highs),
        }
    )
    print(
        f"{n:>8} {widths.mean():>12.4f} {widths.std():>12.4f} "
        f"{np.ptp(lows):>12.4f} {np.ptp(highs):>12.4f}"
    )

In [ ]:
# PRIMARY TEST: CLUSTER BOOTSTRAP (resample cameras)
#
# Rationale: images from the same camera are spatially correlated.
# Image-level bootstrap treats every image as iid and underestimates
# uncertainty. Cluster bootstrap resamples deployment locations with
# replacement, giving properly wider CIs that reflect how performance
# would vary across different sets of camera locations.

qual_mask = np.isin(all_results[MODEL_A]["locations"], list(qualifying_cams))

filtered_results = {}
for model_name, res in all_results.items():
    filtered_results[model_name] = dict(
        targets=res["targets"][qual_mask],
        preds=res["preds"][qual_mask],
        locations=res["locations"][qual_mask],
    )

n_qual_cams = len(np.unique(filtered_results[MODEL_A]["locations"]))
n_qual_imgs = qual_mask.sum()
print(f"Filtered to {n_qual_cams} qualifying cameras ({n_qual_imgs} images)\n")


def cluster_bootstrap_f1(targets, preds, locations, n_boot=N_BOOTSTRAP):
    """
    Bootstrap by resampling cameras (clusters) with replacement.
    Returns: (point_estimate, ci_95, boot_samples)

    Bootstrapping Clustered Data by C. A. Field and A. H. Welsh
    """
    cameras = np.unique(locations)
    cam_idx = {cam: np.where(locations == cam)[0] for cam in cameras}

    boot_f1 = []
    for _ in range(n_boot):
        sampled_cams = rng.choice(cameras, size=len(cameras), replace=True)
        idx = np.concatenate([cam_idx[c] for c in sampled_cams])
        boot_f1.append(
            f1_score(targets[idx], preds[idx], average="macro", zero_division=0)
        )

    boot_f1 = np.array(boot_f1)
    point = f1_score(targets, preds, average="macro", zero_division=0)
    ci = np.percentile(boot_f1, [2.5, 97.5])
    return point, ci, boot_f1


print(f"Cluster bootstrap: {N_BOOTSTRAP} resamples of {n_qual_cams} cameras\n")
clust_results = {}

for model_name, res in filtered_results.items():
    point, ci, samples = cluster_bootstrap_f1(
        res["targets"], res["preds"], res["locations"]
    )
    clust_results[model_name] = dict(point=point, ci=ci, samples=samples)
    print(f"{model_name}")
    print(
        f"  F1 macro: {point:.4f}  95% CI [{ci[0]:.4f}, {ci[1]:.4f}]  width={ci[1] - ci[0]:.4f}"
    )

In [ ]:
# SECONDARY: IMAGE-LEVEL BOOTSTRAP (shown for comparison)
# Narrower CIs because it ignores camera-level correlation.
# Uses the same filtered (qualifying-camera) population as the cluster bootstrap
# above, so the comparison isolates the effect of resampling unit (camera vs.
# image) rather than differing underlying data.

img_boot_results = {}
for model_name, res in filtered_results.items():
    n = len(res["targets"])
    samples = []
    for _ in range(N_BOOTSTRAP):
        idx = rng.integers(0, n, size=n)
        samples.append(
            f1_score(
                res["targets"][idx], res["preds"][idx], average="macro", zero_division=0
            )
        )
    samples = np.array(samples)
    point = f1_score(res["targets"], res["preds"], average="macro", zero_division=0)
    ci = np.percentile(samples, [2.5, 97.5])
    img_boot_results[model_name] = dict(point=point, ci=ci, samples=samples)

print("CI width comparison  (cluster vs image-level bootstrap):")
print(f"{'Model':<25}  {'Cluster CI width':>18}  {'Image CI width':>15}  {'Ratio':>7}")
print("-" * 72)
for mname in MODELS:
    cw = clust_results[mname]["ci"][1] - clust_results[mname]["ci"][0]
    iw = img_boot_results[mname]["ci"][1] - img_boot_results[mname]["ci"][0]
    print(f"{mname:<25}  {cw:>18.4f}  {iw:>15.4f}  {cw / iw:>6.2f}x")

In [ ]:
plt.rcParams.update({"font.family": "serif", "font.size": 9})

# PAPER-READY FIGURE: Methodological justification (Spanish)
# Justifies: camera threshold, bootstrap size, cluster approach.

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4))

# Recompute across all models for generality
bin_edges = [1, 5, 10, 15, 20, 30, 50, 100, np.inf]
bin_centers = [2.5, 7, 12, 17, 25, 40, 75, 150]

model_colors = {
    "DINOv2 ViT-B/14": "#1f77b4",
    "ResNet50 pretrained": "#ff7f0e",
    "timm ViT-B/16": "#2ca02c",
}

for mname in MODELS:
    r = all_results[mname]
    cam_sizes_m, cam_f1s_m = [], []
    for cam in cam_counts:
        mask = r["locations"] == cam
        n = mask.sum()
        if n == 0:
            continue
        f1 = f1_score(
            r["targets"][mask], r["preds"][mask], average="macro", zero_division=0
        )
        cam_sizes_m.append(n)
        cam_f1s_m.append(f1)
    cam_sizes_m = np.array(cam_sizes_m)
    cam_f1s_m = np.array(cam_f1s_m)

    stds = []
    for i in range(len(bin_edges) - 1):
        mask = (cam_sizes_m >= bin_edges[i]) & (cam_sizes_m < bin_edges[i + 1])
        stds.append(cam_f1s_m[mask].std() if mask.sum() > 1 else np.nan)

    short = mname.split()[0]
    ax1.plot(
        bin_centers,
        stds,
        "o-",
        label=short,
        color=model_colors[mname],
        markersize=5,
        linewidth=1.5,
        alpha=0.85,
    )

ax1.axvline(
    MIN_CAM_IMAGES,
    color="red",
    linestyle="--",
    linewidth=1.5,
    alpha=0.7,
    label=f"Umbral = {MIN_CAM_IMAGES}",
)
ax1.set_xlabel("Imágenes por cámara", fontsize=10)
ax1.set_ylabel("Desviación estándar de F1", fontsize=10)
ax1.set_title("(a) Variabilidad F1 por tamaño\nde cámara", fontsize=10)
ax1.set_xscale("log")
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# Uses convergence_data from the convergence cell above
ns = [d["n"] for d in convergence_data]
widths = [d["width"] for d in convergence_data]
ax2.plot(ns, widths, "o-", color="steelblue", markersize=5, linewidth=1.5)
ax2.axvline(
    2000, color="red", linestyle="--", linewidth=1.5, alpha=0.7, label="$B = 2{,}000$"
)
ax2.axhline(widths[-1], color="gray", linestyle=":", alpha=0.5)
ax2.set_xlabel("Remuestreos bootstrap ($B$)", fontsize=10)
ax2.set_ylabel("Ancho del IC 95%", fontsize=10)
ax2.set_title("(b) Convergencia del intervalo\nde confianza", fontsize=10)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, max(widths) * 1.1)

model_names = list(MODELS.keys())
x = np.arange(len(model_names))
width = 0.35

cluster_widths = [
    clust_results[m]["ci"][1] - clust_results[m]["ci"][0] for m in model_names
]
image_widths = [
    img_boot_results[m]["ci"][1] - img_boot_results[m]["ci"][0] for m in model_names
]
ratios = [cluster_widths[i] / image_widths[i] for i in range(len(model_names))]

bars1 = ax3.bar(
    x - width / 2,
    cluster_widths,
    width,
    label="Conglomerados",
    color="steelblue",
    alpha=0.85,
)
bars2 = ax3.bar(
    x + width / 2, image_widths, width, label="Imagen", color="coral", alpha=0.85
)

# Annotate ratio
for i in range(len(model_names)):
    ax3.text(
        x[i],
        max(cluster_widths[i], image_widths[i]) + 0.001,
        f"×{ratios[i]:.1f}",
        ha="center",
        fontsize=9,
        fontweight="bold",
    )

short_names = [m.split()[0] for m in model_names]
ax3.set_xticks(x)
ax3.set_xticklabels(short_names, fontsize=9)
ax3.set_ylabel("Ancho del IC 95%", fontsize=10)
ax3.set_title("(c) Bootstrap por conglomerados\nvs. por imagen", fontsize=10)
ax3.legend(fontsize=9)
ax3.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "methodological_justification.png", dpi=300, bbox_inches="tight"
)
plt.savefig(OUTPUT_DIR / "methodological_justification.pdf", bbox_inches="tight")
plt.show()
print(f"Saved to {OUTPUT_DIR}/methodological_justification.{{png,pdf}}")

# Ratio is computed from the actual results above (not hardcoded) so the caption
# never drifts out of sync with the figure.
ratio_range = f"{min(ratios):.1f}--{max(ratios):.1f}"

print(
    r"""
% ── LaTeX inclusion ──────────────────────────────────────────
\begin{figure}[t]
  \centering
  \includegraphics[width=\linewidth]{figures/methodological_justification.pdf}
  \caption{Justificación de las decisiones metodológicas para la
    inferencia estadística.
    (a)~La desviación estándar del macro-F1 por cámara decrece
    conforme aumenta el número de imágenes y se estabiliza a partir
    de $\sim$15; se adoptó este valor como umbral mínimo.
    (b)~El ancho del intervalo de confianza al 95\% converge a
    partir de $B \approx 500$ remuestreos; se emplearon $B = 2{,}000$.
    (c)~El bootstrap por conglomerados produce intervalos
    $\sim$"""
    + ratio_range
    + r"""$\times$ más anchos que el bootstrap por imagen,
    reflejando la correlación intra-cámara y justificando su uso
    como prueba primaria.}
  \label{fig:methodological_justification}
\end{figure}
"""
)